# Carregamento e pré-processamento do Fake.br

Carrega o corpus do Google Drive, acrescenta `clean_text` e só depois divide treino e teste. A mesma partição guarda `texto_completo` para o BERTimbau e `clean_text` para os modelos tradicionais.

## Preparação do ambiente

Disponibilize a pasta `src` neste ambiente do Colab e monte o Google Drive para acessar o corpus e gravar o resultado processado.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Dependências e recursos de NLP

In [8]:
%pip install -q pandas nltk spacy scikit-learn

import nltk
import spacy
import subprocess
import sys

for recurso in ('corpora/stopwords', 'corpora/wordnet'):
    try:
        nltk.data.find(recurso)
    except LookupError:
        pacote = recurso.rsplit('/', 1)[-1]
        nltk.download(pacote)

if not spacy.util.is_package('pt_core_news_sm'):
    subprocess.run(
        [sys.executable, '-m', 'spacy', 'download', 'pt_core_news_sm'],
        check=True,
    )

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


## Carregamento do corpus

Lê `fake.br-full_texts.csv` e valida as colunas de texto e rótulo antes de iniciar o processamento.

In [9]:
from pathlib import Path
from src.data import carregar_corpus

DATASET_DIR = Path('/content/drive/MyDrive/Fake.br-Corpus')
CSV_PATH = DATASET_DIR / 'fake.br-full_texts.csv'

df = carregar_corpus(CSV_PATH)

# print(f'Corpus carregado: {CSV_PATH}')
# print(f'Registros: {len(df):,}')
# print(df.head())

## Pré-processamento e divisão

A limpeza acrescenta `clean_text` em cada notícia e preserva `texto_completo`. A divisão estratificada vem em seguida e usa as mesmas notícias para os dois paradigmas. O BERTimbau lê o texto original; os modelos tradicionais leem o texto limpo.

In [10]:
from src.data import dividir_corpus, preprocessar_corpus

df = preprocessar_corpus(df)
X_train, X_test, y_train, y_test = dividir_corpus(df)

# print(f'Treino: {len(X_train):,} | Teste: {len(X_test):,}')
# print(df[['texto_completo', 'clean_text', 'label']].head())

## Salvamento da partição

Grava `splits/treino.csv` e `splits/teste.csv` com o texto original, o texto limpo e o rótulo. O treinamento e a avaliação leem esses arquivos e não refazem a divisão. `fake.br-preprocessed.csv` guarda o corpus inteiro com as duas colunas de texto.

In [11]:
from src.data import salvar_split

salvar_split(DATASET_DIR / 'splits', df, X_train, X_test)

PROCESSED_CSV_PATH = DATASET_DIR / 'fake.br-preprocessed.csv'
df.to_csv(PROCESSED_CSV_PATH, index=False)

# print(f'Partição salva em: {DATASET_DIR / "splits"}')
# print(f'Corpus com as duas colunas de texto: {PROCESSED_CSV_PATH}')